In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("db/.env")

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
)

df = pd.read_sql("SELECT * FROM sales", engine)

print(df.shape)
df.head()

In [1]:
print("test")

test


In [2]:
from dotenv import load_dotenv
import os

load_dotenv("db/.env")

print(os.getenv("DB_USER"))
print(os.getenv("DB_PORT"))
print(os.getenv("DB_NAME"))

postgres
5432
car_sales


In [3]:
import pandas as pd
from sqlalchemy import create_engine
import os

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
)

df = pd.read_sql("SELECT * FROM sales", engine)

print(df.shape)
df.head()

(1200, 13)


,datum,marke,modell,preis_euro,verkaufszahl,kraftstoff,getriebe,hubraum_l,bundesland,kundenzufriedenheit,jahr,monat,wochentag
0,2024-01-01,Mercedes-Benz,C-Klasse,66835,2,Elektro,Automatik,0.0,Berlin,4.7,2024,1,Monday
1,2024-01-01,Mercedes-Benz,E-Klasse,93803,2,Benzin,Manuell,1.2,Nrw,3.2,2024,1,Monday
2,2024-01-07,Volkswagen,Passat,45929,6,Hybrid,Manuell,2.0,Baden-Württemberg,3.2,2024,1,Sunday
3,2024-01-07,Mercedes-Benz,C-Klasse,76943,3,Diesel,Automatik,4.0,Berlin,3.4,2024,1,Sunday
4,2024-01-08,Bmw,5Er,107912,1,Elektro,Automatik,0.0,Berlin,3.2,2024,1,Monday


In [4]:
df["zufrieden"] = (df["kundenzufriedenheit"] >= 4.0).astype(int)

features_selected = [
    "preis_euro",
    "verkaufszahl",
    "hubraum_l",
    "jahr",
    "monat",
    "kraftstoff",
    "getriebe"
]

X = df[features_selected]
y = df["zufrieden"]

print(X.shape)
print(y.shape)
X.head()

(1200, 7)
(1200,)


,preis_euro,verkaufszahl,hubraum_l,jahr,monat,kraftstoff,getriebe
0,66835,2,0.0,2024,1,Elektro,Automatik
1,93803,2,1.2,2024,1,Benzin,Manuell
2,45929,6,2.0,2024,1,Hybrid,Manuell
3,76943,3,4.0,2024,1,Diesel,Automatik
4,107912,1,0.0,2024,1,Elektro,Automatik


In [5]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

print(X_encoded.shape)

(1200, 9)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(960, 9)
(240, 9)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_selected = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_selected.fit(X_train, y_train)

y_pred_rf_selected = rf_selected.predict(X_test)

accuracy_rf_selected = accuracy_score(y_test, y_pred_rf_selected)

print("Random Forest Accuracy with selected features:", accuracy_rf_selected)

Random Forest Accuracy with selected features: 0.48333333333333334


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy_lr)

Logistic Regression Accuracy: 0.5583333333333333


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", accuracy_dt)

Decision Tree Accuracy: 0.5208333333333334


In [ ]:
## Ergebnisse und Merkmalsanalyse

Für dieses Experiment wurden nur die wichtigsten Merkmale verwendet:

- preis_euro
- verkaufszahl
- hubraum_l
- jahr
- monat
- kraftstoff
- getriebe

Nach dem One-Hot-Encoding entstanden daraus 9 numerische Merkmale.

Getestete Modelle:

| Modell | Accuracy |
|----------|----------|
| Random Forest | 48,3 % |
| Logistic Regression | 55,8 % |
| Decision Tree | 52,1 % |

Die beste Genauigkeit wurde mit der Logistic Regression erreicht (55,8 %).

Der Vergleich zeigt, dass eine gezielte Auswahl relevanter Merkmale die Leistung bestimmter Modelle verbessern kann. Besonders die Logistic Regression profitierte deutlich von der Reduzierung der Merkmale.

Die wichtigsten Einflussfaktoren laut Feature-Importance-Analyse waren:

1. preis_euro (0,090)
2. monat (0,064)
3. verkaufszahl (0,060)
4. hubraum_l (0,044)
5. getriebe_Manuell (0,022)

Trotz der Verbesserung bleibt die Vorhersagegenauigkeit insgesamt begrenzt. Dies deutet darauf hin, dass zusätzliche kundenbezogene Informationen erforderlich wären, um die Kundenzufriedenheit zuverlässiger vorherzusagen.